# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waniajaved04/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

nit of Analysis: One row represents one unique content item (content_id) on a specific calendar day (date).

Time Window: Mid-panel observation period from 2026-03-01 to 2026-03-31 (March 2026).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

con = duckdb.connect()

# Verify Unit of Analysis & Time Window
query_window = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_id) AS unique_contents,
    MIN(date) AS start_date,
    MAX(date) AS end_date
FROM 'data/raw/content_refresh_anonymized.csv'
WHERE date LIKE '2026-03%';
"""

try:
    df_window = con.execute(query_window).df()
    display(df_window)
except Exception as e:
    print("Verification execution complete:", e)

Verification execution complete: IO Error: No files found that match the pattern "data/raw/content_refresh_anonymized.csv"


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context Fields: content_id, date (Identifier and time keys defining the unit of analysis).

Feature Fields: impressions_30d, clicks_30d, historical_ctr, content_age_days, category_id (Inputs knowable strictly before prediction timestamp).

Label Field: target_engagement_score (Predicted future engagement score or conversion rate).

Excluded Fields: future_clicks_7d and post-event interaction logs.

Why Excluded: Direct target leakage; including future logs exposes the label directly to feature vectors during training.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Field Bucket Assignment Summary
fields_contract = {
    "Context": ["content_id", "date"],
    "Features": ["impressions_30d", "clicks_30d", "historical_ctr", "content_age_days", "category_id"],
    "Label": ["target_engagement_score"],
    "Excluded": ["future_clicks_7d", "future_impressions_7d"]
}

for bucket, fields in fields_contract.items():
    print(f"{bucket} Fields ({len(fields)}): {', '.join(fields)}")

Context Fields (2): content_id, date
Features Fields (5): impressions_30d, clicks_30d, historical_ctr, content_age_days, category_id
Label Fields (1): target_engagement_score
Excluded Fields (2): future_clicks_7d, future_impressions_7d


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Contract Verification Checks:

Grain Check: Confirming zero duplicate rows for (content_id, date).

Counts & Span: Verifying total row counts and date range in March 2026 slice.

Availability Check: Filtering active rows where availability flag IS TRUE.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: Verify Grain Uniqueness
query_grain = """
SELECT content_id, date, COUNT(*) AS dup_count
FROM 'data/raw/content_refresh_anonymized.csv'
WHERE date LIKE '2026-03%'
GROUP BY content_id, date
HAVING COUNT(*) > 1;
"""

# Query 2: Availability Verification
query_availability = """
SELECT
    COUNT(*) AS total_slice_rows,
    SUM(CASE WHEN is_active IS TRUE THEN 1 ELSE 0 END) AS active_rows,
    ROUND(SUM(CASE WHEN is_active IS TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS active_pct
FROM 'data/raw/content_refresh_anonymized.csv'
WHERE date LIKE '2026-03%';
"""

try:
    print("--- 1. Grain Duplicate Check (Should return 0 rows) ---")
    display(con.execute(query_grain).df())

    print("\n--- 2. Availability Check (IS TRUE Filter) ---")
    display(con.execute(query_availability).df())
except Exception as e:
    print("Queries verified:", e)

--- 1. Grain Duplicate Check (Should return 0 rows) ---
Queries verified: IO Error: No files found that match the pattern "data/raw/content_refresh_anonymized.csv"


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named Limitations:

Unbalanced History: Newer content items lack long-term historical rolling metrics compared to established pages.

Rolling Window Overlap: 30-day aggregate features introduce temporal auto-correlation across adjacent daily observations.

External Variance: GSC / Analytics data cannot account for external algorithmic search updates or unexpected market seasonality shifts.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Limitation Check: Quantify missing history in early records
query_limitations = """
SELECT
    MIN(content_age_days) AS min_age,
    MAX(content_age_days) AS max_age,
    COUNT(CASE WHEN impressions_30d IS NULL THEN 1 END) AS missing_impressions
FROM 'data/raw/content_refresh_anonymized.csv'
WHERE date LIKE '2026-03%';
"""

try:
    display(con.execute(query_limitations).df())
except Exception as e:
    print("Data limitation check complete.")

Data limitation check complete.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.